## BASELINES

In [1]:
import numpy as np
import pandas as pd
from utils import MLUtils

from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut, KFold

**Examine training data**

In [2]:
# constants across steps
SEED = 42

In [3]:
de_train_path = "/Users/jennyng/Desktop/Projects/kaggle/single-cell-perturbations/data/de_train.parquet"
de_train = pd.read_parquet(de_train_path)
de_train.head()

,cell_type,sm_name,sm_lincs_id,SMILES,control,A1BG,A1BG-AS1,A2M,A2M-AS1,A2MP1,...,ZUP1,ZW10,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11B,ZYX,ZZEF1
0,NK cells,Clotrimazole,LSM-5341,Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1,False,0.104720,-0.077524,-1.625596,-0.144545,0.143555,...,-0.227781,-0.010752,-0.023881,0.674536,-0.453068,0.005164,-0.094959,0.034127,0.221377,0.368755
1,T cells CD4+,Clotrimazole,LSM-5341,Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1,False,0.915953,-0.884380,0.371834,-0.081677,-0.498266,...,-0.494985,-0.303419,0.304955,-0.333905,-0.315516,-0.369626,-0.095079,0.704780,1.096702,-0.869887
2,T cells CD8+,Clotrimazole,LSM-5341,Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1,False,-0.387721,-0.305378,0.567777,0.303895,-0.022653,...,-0.119422,-0.033608,-0.153123,0.183597,-0.555678,-1.494789,-0.213550,0.415768,0.078439,-0.259365
3,T regulatory cells,Clotrimazole,LSM-5341,Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1,False,0.232893,0.129029,0.336897,0.486946,0.767661,...,0.451679,0.704643,0.015468,-0.103868,0.865027,0.189114,0.224700,-0.048233,0.216139,-0.085024
4,NK cells,Mometasone Furoate,LSM-3349,C[C@@H]1C[C@H]2[C@@H]3CCC4=CC(=O)C=C[C@]4(C)[C...,False,4.290652,-0.063864,-0.017443,-0.541154,0.570982,...,0.758474,0.510762,0.607401,-0.123059,0.214366,0.487838,-0.819775,0.112365,-0.122193,0.676629


In [4]:
de_train.shape

(614, 18216)

In [5]:
de_train.cell_type.value_counts()

cell_type
NK cells              146
T cells CD4+          146
T regulatory cells    146
T cells CD8+          142
B cells                17
Myeloid cells          17
Name: count, dtype: int64

In [6]:
de_train.sm_name.value_counts()

sm_name
R428                   6
MLN 2238               6
Dactolisib             6
LDN 193189             6
Porcn Inhibitor III    6
                      ..
Masitinib              4
GSK-1070916            4
Isoniazid              4
Resminostat            3
CGP 60474              3
Name: count, Length: 146, dtype: int64

In [7]:
de_train.control.value_counts()

control
False    602
True      12
Name: count, dtype: int64

**Establish Baselines**

Return the average gene expression across cell type.

In [8]:
ORIGINAL_FEATURES = ['cell_type', 'sm_name', 'sm_lincs_id','SMILES','control']

In [9]:
gene_cols = [c for c in de_train.columns if c not in ORIGINAL_FEATURES]

In [10]:
def split_data(df, X_features=ORIGINAL_FEATURES, y_cols = gene_cols):
    X_tr = df[ORIGINAL_FEATURES]
    y_tr = df[gene_cols]
    return X_tr, y_tr

In [11]:
# implement MeanBaseline
class MeanBaseline:
    def fit(self, X, y):
        self.means = y.mean()   # average gene expression PER GENE

    def predict(self, X):
        df = pd.DataFrame([self.means] * X.shape[0])
        return df

# implement MedianBaseline
class MedianBaseline:
    def fit(self, X, y):
        self.median = y.median()    # median gene expression PER GENE

    def predict(self, X):
        df = pd.DataFrame([self.median] * X.shape[0])
        return df

# implement ConditionalMeanBaseline
class ConditionalMeanBaseline:
    def __init__(self, condition='sm_name'):
        self.condition = condition

    def fit(self, X, y):
        if self.condition not in X.columns:
            raise ValueError("condition must be in training set </3")
        self.conditional_mean = {} # initialize lookup table category -> mean

        # Get unique categories in `condition`
        u_cat = X[self.condition].unique()

        # For each category, get the mean of that category across samples
        for cat in u_cat:
            is_cat = X[self.condition].values == cat
            cat_mean = y.values[is_cat].mean(axis=0, keepdims=True) #(1, N)
            self.conditional_mean[cat] = cat_mean
        # print(self.conditional_mean.keys())

    def predict(self, X):
        if self.condition not in X.columns:
            raise ValueError("condition must be in test set")

        default_shape = list(self.conditional_mean.values())[0].size
        # print(default_shape)
        y_hat = []

        for cat in X[self.condition].values:
            #  print(cat)
            if cat not in self.conditional_mean.keys():
                # print(f'{cat} not in training set')
                y_hat.append(np.zeros(default_shape))
            else: 
                # print(list(self.conditional_mean[cat]))
                y_hat.append(list(self.conditional_mean[cat]))
        # now u finish :)
        df = pd.DataFrame(np.vstack(y_hat))
        return df


In [12]:
X_tr, y_tr = split_data(de_train)

In [13]:
strat_labels = de_train["sm_name"]
skf = StratifiedKFold(shuffle=True, random_state=SEED)
skf_folds = list(skf.split(X_tr, strat_labels))

/Users/jennyng/miniforge3/envs/bioml/lib/python3.11/site-packages/sklearn/model_selection/_split.py:812: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(


In [14]:
# predict by returning mean gene expression, stratified by drug
mean_model = MeanBaseline()
MLUtils.train(X_tr, y_tr, mean_model, skf_folds)

{'folds': [1.4034054644658713,
  1.348595597505675,
  1.381639670087728,
  1.2662860216956864,
  1.2732128796313242],
 'mean': 1.334627926677257}

In [15]:
# predict by returning median gene expression, stratified by drug
median_model = MedianBaseline()
MLUtils.train(X_tr, y_tr, median_model, skf_folds)

{'folds': [1.3617642037778048,
  1.2884963242776664,
  1.3218213095974074,
  1.151314152944591,
  1.2062543105241406],
 'mean': 1.265930060224322}

In [16]:
conditional_sm_model = ConditionalMeanBaseline('sm_name')
MLUtils.train(X_tr, y_tr, conditional_sm_model, skf_folds)

{'folds': [1.446607062739374,
  1.2405699303632445,
  1.351714515885192,
  1.2500966928268094,
  1.2697978733630997],
 'mean': 1.3117572150355439}

In [17]:
conditional_cell_model = ConditionalMeanBaseline('cell_type')
MLUtils.train(X_tr, y_tr, conditional_sm_model, skf_folds)

{'folds': [1.446607062739374,
  1.2405699303632445,
  1.351714515885192,
  1.2500966928268094,
  1.2697978733630997],
 'mean': 1.3117572150355439}

**Leave One Group Out**

In [18]:
hold_out = ["B cells", "Myeloid cells"]
de_train["cell_type"].isin(hold_out)

0      False
1      False
2      False
3      False
4      False
       ...  
609    False
610    False
611    False
612    False
613    False
Name: cell_type, Length: 614, dtype: bool

In [19]:
subset = de_train[de_train["cell_type"].isin(hold_out)]
subset.head()

,cell_type,sm_name,sm_lincs_id,SMILES,control,A1BG,A1BG-AS1,A2M,A2M-AS1,A2MP1,...,ZUP1,ZW10,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11B,ZYX,ZZEF1
8,B cells,Idelalisib,LSM-1205,CC[C@H](Nc1ncnc2[nH]cnc12)c1nc2cccc(F)c2c(=O)n...,False,0.394173,-0.153824,0.178232,0.566241,0.391377,...,-1.052302,-1.176587,-1.220291,-0.278944,-0.095066,1.101790,0.061803,1.406335,-0.264996,-0.119743
9,Myeloid cells,Idelalisib,LSM-1205,CC[C@H](Nc1ncnc2[nH]cnc12)c1nc2cccc(F)c2c(=O)n...,False,0.025146,0.316388,1.366885,1.301593,2.317619,...,-0.902546,-1.445523,0.794772,0.428973,0.605834,0.271988,0.492231,0.354721,1.471559,-0.259483
30,B cells,Crizotinib,LSM-1027,C[C@@H](Oc1cc(-c2cnn(C3CCNCC3)c2)cnc1N)c1c(Cl)...,False,-0.221234,-0.588647,0.021279,1.104213,1.584397,...,-0.363383,0.215196,1.177265,0.832644,0.296236,0.603842,1.152725,0.599055,0.253165,0.762857
31,Myeloid cells,Crizotinib,LSM-1027,C[C@@H](Oc1cc(-c2cnn(C3CCNCC3)c2)cnc1N)c1c(Cl)...,False,1.322002,-0.083050,1.630350,0.238302,8.488463,...,-1.610365,-0.624159,-0.977755,-1.992494,1.265720,0.461353,-0.270218,-0.077017,-0.508337,0.454613
72,B cells,Linagliptin,LSM-45916,CC#CCn1c(N2CCC[C@@H](N)C2)nc2c1c(=O)n(Cc1nc(C)...,False,-0.096688,0.057827,0.261704,0.054919,1.010873,...,-0.130766,-0.990278,-1.384846,0.742082,-0.349105,-0.091224,-0.721902,-0.098806,-1.570230,-0.013876


In [20]:
subset["sm_name"].value_counts()

sm_name
Idelalisib                  2
Belinostat                  2
Oprozomib (ONX 0912)        2
O-Demethylated Adapalene    2
Dactolisib                  2
Penfluridol                 2
MLN 2238                    2
Foretinib                   2
Porcn Inhibitor III         2
Crizotinib                  2
R428                        2
LDN 193189                  2
Alvocidib                   2
Dabrafenib                  2
Palbociclib                 2
Linagliptin                 2
CHIR-99021                  2
Name: count, dtype: int64

In [21]:
groups = de_train['cell_type'].to_numpy()

In [22]:
MLUtils.train_logo(X_tr, y_tr, groups, mean_model)

hold out group B cells: MRRMSE = 2.9578107045881303
hold out group Myeloid cells: MRRMSE = 3.4373698915674664
hold out group NK cells: MRRMSE = 1.4064921355194135
hold out group T cells CD4+: MRRMSE = 1.3807927571315313
hold out group T cells CD8+: MRRMSE = 1.0604447160678676
hold out group T regulatory cells: MRRMSE = 1.1804720987765625


{'folds': [2.9578107045881303,
  3.4373698915674664,
  1.4064921355194135,
  1.3807927571315313,
  1.0604447160678676,
  1.1804720987765625],
 'mean': 1.9038970506084951}

In [23]:
MLUtils.train_logo(X_tr, y_tr, groups, median_model)

hold out group B cells: MRRMSE = 2.9859827096724394
hold out group Myeloid cells: MRRMSE = 3.498583703107072
hold out group NK cells: MRRMSE = 1.3711043900090336
hold out group T cells CD4+: MRRMSE = 1.2524717680931248
hold out group T cells CD8+: MRRMSE = 0.8360167774031555
hold out group T regulatory cells: MRRMSE = 1.1607833240064853


{'folds': [2.9859827096724394,
  3.498583703107072,
  1.3711043900090336,
  1.2524717680931248,
  0.8360167774031555,
  1.1607833240064853],
 'mean': 1.8508237787152186}

In [24]:
MLUtils.train_logo(X_tr, y_tr, groups, conditional_cell_model)

hold out group B cells: MRRMSE = 3.012334628347464
hold out group Myeloid cells: MRRMSE = 3.5249428292507456
hold out group NK cells: MRRMSE = 1.3852749270962121
hold out group T cells CD4+: MRRMSE = 1.2595409565631586
hold out group T cells CD8+: MRRMSE = 0.8276923177113612
hold out group T regulatory cells: MRRMSE = 1.1781031338973962


{'folds': [3.012334628347464,
  3.5249428292507456,
  1.3852749270962121,
  1.2595409565631586,
  0.8276923177113612,
  1.1781031338973962],
 'mean': 1.8646481321443897}

In [25]:
MLUtils.train_logo(X_tr, y_tr, groups, conditional_sm_model)

hold out group B cells: MRRMSE = 2.1996774575520512
hold out group Myeloid cells: MRRMSE = 2.913299763127571
hold out group NK cells: MRRMSE = 1.2414821891170322
hold out group T cells CD4+: MRRMSE = 1.192783811268897
hold out group T cells CD8+: MRRMSE = 1.1311151585594748
hold out group T regulatory cells: MRRMSE = 1.385659771090541


{'folds': [2.1996774575520512,
  2.913299763127571,
  1.2414821891170322,
  1.192783811268897,
  1.1311151585594748,
  1.385659771090541],
 'mean': 1.6773363584525949}

**PICKLE SPLITZZ**

Re-assessing baselines using new splits

In [26]:
import pickle

In [27]:
path = "/Users/jennyng/Desktop/Projects/kaggle/single-cell-perturbations/processed_data/train_test_splits.pkl"
with open(path, "rb") as f:
    folds = pickle.load(f)

In [28]:
MLUtils.train(X_tr, y_tr, mean_model, folds)

{'folds': [1.3209598271842464,
  1.1868509100361975,
  1.6175502815986589,
  1.3510822804390379,
  1.7410636875131322,
  1.1744338322718781,
  1.016755056838596,
  1.0726439532047967,
  1.229475584504458,
  1.6165721045585022],
 'mean': 1.3327387518149503}

In [29]:
MLUtils.train(X_tr, y_tr, median_model, folds)

{'folds': [1.2437315041715464,
  1.122980624195546,
  1.5520098978253174,
  1.258750698148323,
  1.722393451609928,
  1.0980687894065877,
  0.9350838624715622,
  0.9902826697830833,
  1.1337707196710751,
  1.5820665101797793],
 'mean': 1.2639138727462746}

In [30]:
MLUtils.train(X_tr, y_tr, conditional_cell_model, folds)

{'folds': [1.3435588269870102,
  1.1715147277997335,
  1.658911794353903,
  1.3403040689849415,
  1.7265702163933947,
  1.229416875648161,
  1.067378494156077,
  1.1236522212160858,
  1.253438705321743,
  1.593028887620699],
 'mean': 1.350777481848175}

In [31]:
MLUtils.train(X_tr, y_tr, conditional_sm_model, folds)

{'folds': [1.2294358790461617,
  1.228896287081401,
  1.6803462229376818,
  1.3883612059644383,
  1.44081456786079,
  1.2195944001661698,
  1.0517451599333232,
  1.1783288048020233,
  1.3214182734920774,
  1.619463783709407],
 'mean': 1.3358404584993473}